In [3]:
import duckdb
con = duckdb.connect()

In [4]:
# Nearby potential planets query
con.sql("""
    SELECT p.pl_name, s.hostname, s.sy_dist, p.pl_rade, p.pl_eqt
    FROM read_parquet('../data/mart/fact_planets.parquet') p
    JOIN read_parquet('../data/mart/dim_star.parquet') s ON p.star_key = s.star_key
    WHERE s.sy_dist < 15
      AND p.pl_eqt BETWEEN 200 AND 350
    ORDER BY s.sy_dist
""")

┌────────────────────┬──────────────────┬─────────┬─────────┬────────┐
│      pl_name       │     hostname     │ sy_dist │ pl_rade │ pl_eqt │
│      varchar       │     varchar      │ double  │ double  │ double │
├────────────────────┼──────────────────┼─────────┼─────────┼────────┤
│ Proxima Cen b      │ Proxima Cen      │ 1.30119 │    NULL │  218.0 │
│ Proxima Cen d      │ Proxima Cen      │ 1.30119 │    NULL │  282.0 │
│ Barnard e          │ Barnard's star   │ 1.82655 │    NULL │  340.0 │
│ GJ 887 c           │ GJ 887           │ 3.28679 │    NULL │  320.0 │
│ GJ 887 d           │ GJ 887           │ 3.28679 │    NULL │  241.0 │
│ Ross 128 b         │ Ross 128         │ 3.37454 │    NULL │  301.0 │
│ Teegarden's Star b │ Teegarden's Star │ 3.83078 │    NULL │  277.0 │
│ Teegarden's Star c │ Teegarden's Star │ 3.83078 │    NULL │  209.0 │
│ GJ 1002 b          │ GJ 1002          │ 4.84867 │    NULL │  230.9 │
│ GJ 251 b           │ GJ 251           │ 5.58057 │    NULL │  336.0 │
│    ·

In [5]:
# Discoveries by year
con.sql("""
    SELECT dt.disc_year, COUNT(*) as planets_discovered
    FROM read_parquet('../data/mart/fact_planets.parquet') p
    JOIN read_parquet('../data/mart/dim_date.parquet') dt ON p.date_key = dt.date_key
    GROUP BY dt.disc_year
    ORDER BY dt.disc_year
""")

┌───────────┬────────────────────┐
│ disc_year │ planets_discovered │
│   int64   │       int64        │
├───────────┼────────────────────┤
│      1992 │                  2 │
│      1994 │                  1 │
│      1995 │                  1 │
│      1996 │                  6 │
│      1997 │                  1 │
│      1998 │                  6 │
│      1999 │                 13 │
│      2000 │                 16 │
│      2001 │                 12 │
│      2002 │                 29 │
│        ·  │                  · │
│        ·  │                  · │
│        ·  │                  · │
│      2017 │                152 │
│      2018 │                308 │
│      2019 │                196 │
│      2020 │                234 │
│      2021 │                564 │
│      2022 │                368 │
│      2023 │                323 │
│      2024 │                260 │
│      2025 │                245 │
│      2026 │                207 │
├───────────┴────────────────────┤
│ 34 rows (20 shown)

In [6]:
# Detection methods
con.sql("""
    SELECT d.discoverymethod, COUNT(*) as planet_count, ROUND(AVG(p.pl_rade), 2) as avg_radius
    FROM read_parquet('../data/mart/fact_planets.parquet') p
    JOIN read_parquet('../data/mart/dim_discovery.parquet') d ON p.discovery_key = d.discovery_key
    GROUP BY d.discoverymethod
    ORDER BY planet_count DESC
""")

┌───────────────────────────────┬──────────────┬────────────┐
│        discoverymethod        │ planet_count │ avg_radius │
│            varchar            │    int64     │   double   │
├───────────────────────────────┼──────────────┼────────────┤
│ Transit                       │         4652 │       4.34 │
│ Radial Velocity               │         1186 │       4.71 │
│ Microlensing                  │          278 │       NULL │
│ Imaging                       │           97 │      20.94 │
│ Transit Timing Variations     │           41 │       3.89 │
│ Eclipse Timing Variations     │           17 │       NULL │
│ Orbital Brightness Modulation │            9 │       5.62 │
│ Pulsar Timing                 │            8 │       NULL │
│ Astrometry                    │            6 │       NULL │
│ Pulsation Timing Variations   │            2 │       NULL │
│ Disk Kinematics               │            1 │       NULL │
├───────────────────────────────┴──────────────┴────────────┤
│ 11 row

In [7]:
# Most productive facilities
con.sql("""
    SELECT d.disc_facility, COUNT(*) as planet_count
    FROM read_parquet('../data/mart/fact_planets.parquet') p
    JOIN read_parquet('../data/mart/dim_discovery.parquet') d ON p.discovery_key = d.discovery_key
    GROUP BY d.disc_facility
    ORDER BY planet_count DESC
    LIMIT 10
""")

┌──────────────────────────────────────────────┬──────────────┐
│                disc_facility                 │ planet_count │
│                   varchar                    │    int64     │
├──────────────────────────────────────────────┼──────────────┤
│ Kepler                                       │         2784 │
│ Transiting Exoplanet Survey Satellite (TESS) │          896 │
│ K2                                           │          549 │
│ Multiple Observatories                       │          353 │
│ La Silla Observatory                         │          306 │
│ W. M. Keck Observatory                       │          194 │
│ KMTNet                                       │          137 │
│ SuperWASP                                    │          122 │
│ OGLE                                         │          110 │
│ HATSouth                                     │           73 │
├──────────────────────────────────────────────┴──────────────┤
│ 10 rows                               

In [8]:
# Planets that are similar to Earth
con.sql("""
    SELECT p.pl_name, s.hostname, s.sy_dist, p.pl_rade, p.pl_eqt
    FROM read_parquet('../data/mart/fact_planets.parquet') p
    JOIN read_parquet('../data/mart/dim_star.parquet') s ON p.star_key = s.star_key
    WHERE p.pl_rade BETWEEN 0.5 AND 1.5
      AND p.pl_eqt BETWEEN 200 AND 350
    ORDER BY s.sy_dist
""")

┌───────────────┬─────────────┬─────────┬─────────┬────────┐
│    pl_name    │  hostname   │ sy_dist │ pl_rade │ pl_eqt │
│    varchar    │   varchar   │ double  │ double  │ double │
├───────────────┼─────────────┼─────────┼─────────┼────────┤
│ LP 890-9 c    │ LP 890-9    │ 32.4298 │   1.367 │  272.0 │
│ TOI-2095 c    │ TOI-2095    │ 41.9176 │    1.33 │  297.0 │
│ TOI-2095 b    │ TOI-2095    │ 41.9176 │    1.25 │  347.0 │
│ K2-3 d        │ K2-3        │ 44.0727 │   1.458 │  305.2 │
│ Kepler-1649 c │ Kepler-1649 │ 92.1913 │    1.06 │  234.0 │
│ Kepler-1649 b │ Kepler-1649 │ 92.1913 │   1.017 │  307.0 │
│ Kepler-62 f   │ Kepler-62   │ 300.874 │    1.41 │  208.0 │
│ Kepler-1126 c │ Kepler-1126 │ 635.736 │    1.45 │  305.0 │
└───────────────┴─────────────┴─────────┴─────────┴────────┘